# Quantile crossing validation of GPQR models for H

In [ ]:
import sys
import os
import math

import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

sys.path.insert(0, os.path.abspath(".."))

try:
    import config_notebook
except ImportError:
    print("Output will not be deterministic SVG.")

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Load data

In [ ]:
from scripts.crossing import quantile_crossing

QUANTILES = torch.tensor([0.05, 0.25, 0.5, 0.75, 0.95]).to(device)
CENTER_QUANTILE_INDEX = 2
NUM_LOWER_QUANTILES = 2
NUM_LATENTS = 3
NUM_LOWER_LATENTS = 1

N_EPOCHS = int(os.getenv("HEAVYEDGE_N_EPOCHS", 10000))

X_train = pd.read_csv("../_temp/X.csv").drop(columns="Slurry").values
y_train = pd.read_csv("../_temp/y.csv")["H"].values

In [ ]:
scaler = MinMaxScaler().fit(X_train)
X_scale = scaler.scale_
X_min = scaler.min_

In [ ]:
d0 = np.linspace(X_train[:, 0].min(), X_train[:, 0].max(), 10)
d1 = np.linspace(X_train[:, 1].min(), X_train[:, 1].max(), 10)
d2 = np.unique(X_train[:, -1])  # 4 unique values

grid = np.meshgrid(d0, d1, d2, indexing="ij")
X_pred = np.stack(grid, axis=-1).reshape(-1, X_train.shape[-1])

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.float32).to(device)
X_scale = torch.tensor(X_scale, dtype=torch.float32).to(device)
X_min = torch.tensor(X_min, dtype=torch.float32).to(device)
X_pred = torch.tensor(X_pred, dtype=torch.float32).to(device)

## Direct LMC

In [ ]:
from scripts.all_models import DirectLmcMtgpqr_H
from gpytorch_qr.likelihoods import MultitaskQuantileGPLikelihood

QC_PATH = "../_temp/direct_lmc_crossing.H.csv"

if os.path.exists(QC_PATH):
    direct_lmc_qc = pd.read_csv(QC_PATH)
else:
    model = DirectLmcMtgpqr_H(
        inducing_points=X_train.clone().detach(),
        num_quantiles=len(QUANTILES),
        num_lower_quantiles=NUM_LOWER_QUANTILES,
        num_latents=NUM_LATENTS,
        num_lower_latents=NUM_LOWER_LATENTS,
        X_scale=X_scale,
        X_min=X_min,
    ).to(device)
    likelihood = MultitaskQuantileGPLikelihood(
        QUANTILES,
        torch.zeros((len(QUANTILES),)),
        learn_scales=True,
    ).to(device)

    crossing_rate, mean_crossing, max_crossing = quantile_crossing(
        X_train,
        y_train,
        X_pred,
        model,
        likelihood,
        N_EPOCHS,
        learning_rate=0.001,
    )
    direct_lmc_qc = pd.DataFrame(
        {
            "crossing_rate": crossing_rate,
            "mean_crossing": mean_crossing,
            "max_crossing": max_crossing,
        }
    )
    direct_lmc_qc.to_csv(QC_PATH, index=False)

## Direct independent

In [ ]:
from scripts.all_models import DirectIndependentMtgpqr_H
from gpytorch_qr.likelihoods import MultitaskQuantileGPLikelihood

QC_PATH = "../_temp/direct_independent_crossing.H.csv"

if os.path.exists(QC_PATH):
    direct_independent_qc = pd.read_csv(QC_PATH)
else:
    model = DirectIndependentMtgpqr_H(
        inducing_points=X_train.clone().detach(),
        num_quantiles=len(QUANTILES),
        num_lower_quantiles=NUM_LOWER_QUANTILES,
        num_latents=NUM_LATENTS,
        num_lower_latents=NUM_LOWER_LATENTS,
        X_scale=X_scale,
        X_min=X_min,
    ).to(device)
    likelihood = MultitaskQuantileGPLikelihood(
        QUANTILES,
        torch.zeros((len(QUANTILES),)),
        learn_scales=True,
    ).to(device)

    crossing_rate, mean_crossing, max_crossing = quantile_crossing(
        X_train,
        y_train,
        X_pred,
        model,
        likelihood,
        N_EPOCHS,
        learning_rate=0.001,
    )
    direct_independent_qc = pd.DataFrame(
        {
            "crossing_rate": crossing_rate,
            "mean_crossing": mean_crossing,
            "max_crossing": max_crossing,
        }
    )
    direct_independent_qc.to_csv(QC_PATH, index=False)